# EviChange — Phase 14 ground truth (Quang Ninh)

Four label sets for the extended evaluation (decisions.md D13, D14). Like Phase 3, these labels are
ground truth, so **they must not depend on anything the system computes**.

| Task | What you label | Items | Imagery |
|---|---|---|---|
| (a) classes | dominant land-cover class inside the box, before and after | the 30 Phase 3 sites | Sentinel-2 dry-season composites (the Phase 3 chips) |
| (b) shoreline | where the land / water edge crosses the line | 30 transects × 2 dates | PlanetScope, 3 m |
| (c) cells | built or not built, at both dates | 30 coastal cells | PlanetScope, 3 m |
| (d) change month | whether a lasting change happens, and between which months | the 30 sites | PlanetScope, every month |

**Blind protocol (write this in the report's method section):**
1. Label **before** opening any output for the item: no Dynamic World classes, no extracted shorelines,
   no transect rates, no Phase 12 built-up numbers, no CCDC dates. The app hides the classes at the
   labelled sites and the shorelines at the reference transects until these labels are in the repo.
2. The yellow line in (b) is the transect **extended by a random length at each end**, so where the
   line starts and ends says nothing about where the system put the shoreline.
3. In (b) the two dates of a transect come up separately, in shuffled order, so one answer doesn't
   anchor the other.
4. `unsure`, `not visible` and `no edge` are valid answers. Don't guess to fill the form.
5. The labeller built the classifier and the shoreline method (PLAN Phase 14); the write-up says so.

**Before you start:** copy these repo files into `MyDrive/evichange/labeling/` (the Phase 3 folder,
which already has `sites.json` and the chips): `data/quangninh/labels/phase14/reference_transects.json`
and `coastal_cells.json`. The Planet months must be in Earth Engine
(`notebooks/EviChange_planet_to_ee.ipynb`) for tasks (b)–(d); task (a) only needs Sentinel-2. About 2½ hours for all four tasks; progress is saved
after every **Save**, so you can stop and resume.

Run cells top to bottom with Shift+Enter, then run only the task you're doing.

## Step 1 — Connect Earth Engine and Drive

In [ ]:
!pip -q install earthengine-api
import ee, io, json, math, os, random, time, datetime, urllib.request
from google.colab import drive
drive.mount('/content/drive')

EE_PROJECT = 'space-510615'          # the Cloud project registered for Earth Engine
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

ROOT  = '/content/drive/MyDrive/evichange/labeling'
CHIPS = f'{ROOT}/chips'               # Phase 3 chips are reused for task (a)
CHIPS14 = f'{ROOT}/chips14'
os.makedirs(CHIPS, exist_ok=True); os.makedirs(CHIPS14, exist_ok=True)
print('Earth Engine + Drive ready:', ROOT)

## Step 2 — Settings
`LABELER`: `labeler1` labels everything; the second labeller (P3) sets `labeler2` and `N_ITEMS = 10`,
which picks the same 10 items of each task for every second labeller (a fixed seed). Output files are
`labels14_<task>.jsonl` for `labeler1` and `labels14_<task>_<labeler>.jsonl` otherwise.

`REF_DATES` are the two dates of (b) and (c). The plan says Oct 2024 and Sep 2026; Sep 2026 is ordered
in November (D12), so until then the last delivered Planet month stands in, and every record says
which month it used.

In [ ]:
LABELER  = 'labeler1'      # 'labeler2' for the second labeller
N_ITEMS  = None            # None = all items; 10 for the second labeller
SEED     = 20261023        # same as agents/labels14.py
PROTOCOL = 'labels14_v1'
WANT_DATES = ('2024-10', '2026-09')

# Phase 3 / v3 detector windows: task (a) labels the classes of these two dry seasons
BEFORE = ('2018-11-01', '2019-03-31')
AFTER  = ('2022-11-01', '2023-03-31')
AOI_BOUNDS = [107.22, 20.98, 107.40, 21.10]

PLANET = 'projects/space-510615/assets/planet_qn'
LAND_M = 300                # transects start 300 m landward of the baseline (agents/shoreline.py)
M_PER_DEG = 111_320

planet = ee.ImageCollection(PLANET)
try:
    info = planet.reduceColumns(ee.Reducer.toList(3), ['month', 'date', 'clear_share']).get('list').getInfo()
except ee.EEException:              # collection not created yet
    info = []
PLANET_MONTHS = {m: {'date': d, 'clear': c} for m, d, c in sorted(info)}
MONTHS = sorted(PLANET_MONTHS)
if MONTHS:
    REF_DATES = tuple(m if m in PLANET_MONTHS else (MONTHS[0] if i == 0 else MONTHS[-1]) for i, m in enumerate(WANT_DATES))
    print('Planet months:', ', '.join(MONTHS))
    print('reference dates for (b) and (c):', REF_DATES, '' if REF_DATES == WANT_DATES else f'(wanted {WANT_DATES})')
else:
    REF_DATES = ()
    print('No Planet months in Earth Engine yet: task (a) works; (b), (c) and (d) need EviChange_planet_to_ee.ipynb first')

def need_planet():
    assert MONTHS, 'no Planet months in Earth Engine: run EviChange_planet_to_ee.ipynb first (task (a) does not need them)'

sites = json.load(open(f'{ROOT}/sites.json'))
refs  = json.load(open(f'{ROOT}/reference_transects.json'))
cells_ = json.load(open(f'{ROOT}/coastal_cells.json'))

def subset(items):
    return items if not N_ITEMS else random.Random(SEED + 2).sample(items, k=min(N_ITEMS, len(items)))

def out_file(task):
    return f'{ROOT}/labels14_{task}.jsonl' if LABELER == 'labeler1' else f'{ROOT}/labels14_{task}_{LABELER}.jsonl'

print(len(sites), 'sites,', len(refs), 'transects,', len(cells_), 'cells; labeller', LABELER)

## Step 3 — Shared helpers
Chips are fetched once and cached in `chips14/`. Planet chips show the whole mosaic, clouds included
(the clear mask is not applied), so you can judge whether the surface is visible. Overlays are drawn
locally, so the slider in (b) moves the marker without a new download.

In [ ]:
import ipywidgets as W
from IPython.display import display
from PIL import Image, ImageDraw

PLANET_VIS = {'rgb': ([5, 3, 1], 0, 3000), 'nir': ([7, 5, 3], 0, 4500)}   # band positions (agents/planet.py)

def frame(lon0, lat0):
    kx = M_PER_DEG * math.cos(math.radians(lat0))
    return (lambda lon, lat: ((lon - lon0) * kx, (lat - lat0) * M_PER_DEG),
            lambda x, y: (lon0 + x / kx, lat0 + y / M_PER_DEG))

def square(lon, lat, side_m):
    _, to_ll = frame(lon, lat)
    (w, s), (e, n) = to_ll(-side_m / 2, -side_m / 2), to_ll(side_m / 2, side_m / 2)
    return [w, s, e, n]

def planet_chip(month, bbox, kind='rgb', px=512):
    # PNG of one Planet month over bbox in EPSG:4326, so pixels map linearly to lon/lat.
    w, s, e, n = bbox
    path = f"{CHIPS14}/planet_{month}_{kind}_{w:.5f}_{s:.5f}_{e:.5f}_{n:.5f}_{px}.png"
    if not os.path.exists(path):
        bands, lo, hi = PLANET_VIS[kind]
        y, m = month.split('-')
        img = ee.Image(f'{PLANET}/m{y}_{m}').select(bands).visualize(min=lo, max=hi)
        height = round(px * (n - s) / ((e - w) * math.cos(math.radians((s + n) / 2))))
        url = img.unmask(0).getThumbURL({'region': ee.Geometry.Rectangle(bbox, 'EPSG:4326', False),
                                         'dimensions': f'{px}x{height}', 'crs': 'EPSG:4326', 'format': 'png'})
        open(path, 'wb').write(urllib.request.urlopen(url, timeout=120).read())
    return Image.open(path).convert('RGB'), bbox

def to_px(img, bbox, lon, lat):
    w, s, e, n = bbox
    return (lon - w) / (e - w) * img.width, (n - lat) / (n - s) * img.height

def png(img):
    buf = io.BytesIO(); img.save(buf, format='PNG'); return buf.getvalue()

def draw_ring(img, bbox, ring, colour=(255, 255, 0), width=2):
    d = ImageDraw.Draw(img)
    d.line([to_px(img, bbox, *p) for p in ring], fill=colour, width=width)
    return img

def done_keys(path, key):
    return {key(json.loads(l)) for l in open(path)} if os.path.exists(path) else set()

def run_form(items, path, key, build, title):
    # One item at a time. build(item) -> (widgets, collect); collect() returns the task's fields or
    # raises ValueError with what's missing. Confidence, notes, timing and Save are shared.
    todo = [it for it in items if key(it) not in done_keys(path, key)]
    box = W.VBox(); display(box)
    st = {'i': 0}

    def show():
        if st['i'] >= len(todo):
            box.children = [W.HTML(f'<b>{title}: all {len(items)} items labelled.</b> Saved to {path}')]
            return
        it = todo[st['i']]; t0 = time.time()
        parts, collect = build(it)
        conf = W.ToggleButtons(options=['low', 'medium', 'high'], value=None, description='Confidence')
        notes = W.Text(placeholder='notes', layout=W.Layout(width='600px'))
        msg = W.HTML(); save = W.Button(description='Save', button_style='success'); skip = W.Button(description='Skip for now')

        def on_save(_):
            try:
                fields = collect()
                if conf.value is None:
                    raise ValueError('pick a confidence')
            except ValueError as e:
                msg.value = f'<span style="color:#c00">{e}</span>'; return
            rec = {'protocol': PROTOCOL, 'labeler': LABELER, 'labeled_at': datetime.datetime.now().isoformat(timespec='seconds'),
                   **fields, 'confidence': conf.value, 'notes': notes.value.strip(), 'seconds_spent': round(time.time() - t0)}
            with open(path, 'a') as f:
                f.write(json.dumps(rec) + '\n')
            st['i'] += 1; show()

        def on_skip(_):
            todo.append(todo.pop(st['i'])); show()

        save.on_click(on_save); skip.on_click(on_skip)
        head = W.HTML(f'<h3>{title} — {key(it)} ({len(done_keys(path, key)) + 1} of {len(items)})</h3>')
        box.children = [head, *parts, conf, notes, W.HBox([save, skip]), msg]

    show()

## Task (a) — Land-cover class at the 30 sites
Same chips as Phase 3 (true colour and SWIR false colour, before = Nov 2018 – Mar 2019, after =
Nov 2022 – Mar 2023). For each date pick the class that covers **most of the yellow box**. The classes are
Dynamic World's, but you never see Dynamic World:

| class | means |
|---|---|
| water | permanent or seasonal open water, ponds, rivers, sea |
| trees | forest, plantations, mangroves (tall woody cover) |
| grass | open grassland, lawns, golf courses |
| flooded_vegetation | marsh, wet vegetation with water among it (not mangrove trees) |
| crops | fields, paddies |
| shrub_and_scrub | low woody cover, regrowth, scrub on old spoil |
| built | buildings, roads, paved yards, industrial plants, port areas |
| bare | bare soil, rock, sand, mine pits, spoil dumps, coal stockpiles |
| snow_and_ice | (not in Quang Ninh; it's in the list only because Dynamic World has it) |

Bare spoil next to a mine is **bare**, not built, even when there are buildings nearby: this is the
case D10 flags (Dynamic World's most common transition is built → trees).

In [ ]:
S2 = 'COPERNICUS/S2_SR_HARMONIZED'
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
VIS = {'rgb': (['B4', 'B3', 'B2'], 0.0, 0.3), 'swir': (['B12', 'B8', 'B4'], 0.0, 0.45)}
CLASSES = ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice', 'unsure']
aoi = ee.Geometry.Rectangle(AOI_BOUNDS)

def mask_scl(img):
    scl = img.select('SCL')
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return img.select(BANDS).updateMask(clear).divide(10000)

def composite(window):
    return (ee.ImageCollection(S2).filterBounds(aoi).filterDate(*window)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)).map(mask_scl).median().clip(aoi))

COMPOSITES = {'before': composite(BEFORE), 'after': composite(AFTER)}

def s2_chip(site, when, kind):
    # The Phase 3 chip if it is in Drive, else rendered the same way.
    path = f"{CHIPS}/{site['site_id']}_{when}_{kind}.png"
    if not os.path.exists(path):
        bands, lo, hi = VIS[kind]
        edge = (ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(ee.Geometry(site['outline']))]), 1, 2)
                .visualize(palette=['ffff00']))
        url = COMPOSITES[when].visualize(bands=bands, min=lo, max=hi).blend(edge).getThumbURL(
            {'region': site['chip'], 'dimensions': 512, 'format': 'png'})
        open(path, 'wb').write(urllib.request.urlopen(url, timeout=120).read())
    return open(path, 'rb').read()

def build_classes(site):
    imgs = [W.VBox([W.HTML(f'<b>{when}</b> ({kind})'), W.Image(value=s2_chip(site, when, kind), width=330)])
            for when in ('before', 'after') for kind in VIS]
    before = W.Dropdown(options=[None] + CLASSES, description='before')
    after = W.Dropdown(options=[None] + CLASSES, description='after')
    gmaps = W.HTML(f'<a href="https://www.google.com/maps/@{site["lat"]},{site["lon"]},800m/data=!3m1!1e3" target="_blank">'
                   'high-res (today\'s imagery: to recognise surfaces only)</a>')

    def collect():
        if before.value is None or after.value is None:
            raise ValueError('pick both classes')
        return {'site_id': site['site_id'], 'class_before': before.value, 'class_after': after.value,
                'before_window': list(BEFORE), 'after_window': list(AFTER)}
    return [W.HBox(imgs[:2]), W.HBox(imgs[2:]), gmaps, W.HBox([before, after])], collect

run_form(subset(sites), out_file('classes'), lambda r: r['site_id'], build_classes, '(a) classes')

## Task (b) — Reference shorelines on 30 transects
Each item is one transect at one date. Move the slider until the red marker sits where **the land
meets the water along the yellow line**: the wet/dry edge you see in this image (the waterline at the
moment the image was taken). Ticks are every 50 m, numbers every 100 m from the landward end of the
line. True colour and NIR false colour (water is dark in NIR) are side by side.
- `edge`: you can place the edge. If the line crosses several edges (a lagoon, a sand bar), use the
  one between the **open sea** and the first land.
- `no_edge`: the line is all land or all water in this image.
- `not_visible`: cloud, haze or missing data over the edge.
Record the image's date if the notes need it; the form stores the acquisition date anyway.

In [ ]:
def transect_geometry(ref):
    # Drawn line (start, end, length m) = the transect extended by extend_land_m / extend_sea_m,
    # as agents/labels14.drawn_segment().
    (lon_a, lat_a), (lon_b, lat_b) = ref['geometry']['coordinates']
    to_m, to_ll = frame(lon_a, lat_a)
    bx, by = to_m(lon_b, lat_b); length = math.hypot(bx, by); ux, uy = bx / length, by / length
    el, es = ref['extend_land_m'], ref['extend_sea_m']
    return to_ll(-ux * el, -uy * el), to_ll(bx + ux * es, by + uy * es), length + el + es

def along(start, end, length, s):
    t = s / length
    return start[0] + t * (end[0] - start[0]), start[1] + t * (end[1] - start[1])

def draw_transect(img, bbox, start, end, length, marker=None):
    img = img.copy(); d = ImageDraw.Draw(img)
    a, b = to_px(img, bbox, *start), to_px(img, bbox, *end)
    d.line([a, b], fill=(255, 255, 0), width=2)
    nx, ny = (b[1] - a[1]), -(b[0] - a[0]); k = math.hypot(nx, ny) or 1; nx, ny = nx / k, ny / k
    for s in range(0, int(length) + 1, 50):
        x, y = to_px(img, bbox, *along(start, end, length, s)); r = 7 if s % 100 == 0 else 4
        d.line([(x - nx * r, y - ny * r), (x + nx * r, y + ny * r)], fill=(255, 255, 0), width=2)
        if s % 100 == 0:
            d.text((x + nx * 10, y + ny * 10), str(s), fill=(255, 255, 0))
    if marker is not None:
        x, y = to_px(img, bbox, *along(start, end, length, marker))
        d.ellipse([x - 6, y - 6, x + 6, y + 6], outline=(255, 0, 0), width=3)
    return img

need_planet()
shore_items = [{'transect_id': r['transect_id'], 'month': m, 'ref': r} for r in subset(refs) for m in REF_DATES]
random.Random(SEED + 3).shuffle(shore_items)

def build_shoreline(item):
    ref, month = item['ref'], item['month']
    start, end, length = transect_geometry(ref)
    mid = along(start, end, length, length / 2)
    bbox = square(*mid, length + 200)
    base = {k: planet_chip(month, bbox, k, px=480)[0] for k in PLANET_VIS}
    views = {k: W.Image(width=480) for k in PLANET_VIS}
    slider = W.FloatSlider(value=length / 2, min=0, max=round(length), step=5, description='edge at (m)',
                           continuous_update=False, layout=W.Layout(width='600px'))
    status = W.RadioButtons(options=['edge', 'no_edge', 'not_visible'], value=None, description='Result')

    def redraw(*_):
        mark = slider.value if status.value in (None, 'edge') else None
        for k in PLANET_VIS:
            views[k].value = png(draw_transect(base[k], bbox, start, end, length, mark))
    slider.observe(redraw, 'value'); status.observe(redraw, 'value'); redraw()
    date = PLANET_MONTHS[month]['date']

    def collect():
        if status.value is None:
            raise ValueError('pick a result')
        edge = status.value == 'edge'
        return {'transect_id': ref['transect_id'], 'month': month, 'image_date': date, 'status': status.value,
                'drawn_m': slider.value if edge else None,
                'pos_m': round(slider.value - ref['extend_land_m'] - LAND_M, 1) if edge else None}
    return [W.HTML(f'Planet image of <b>{date}</b>'), W.HBox([views['rgb'], views['nir']]), slider, status], collect

run_form(shore_items, out_file('shoreline'), lambda r: f"{r['transect_id']} {r['month']}", build_shoreline, '(b) shoreline')

## Task (c) — Built or not built: 30 coastal cells
The cyan square is a 30 m cell, 0–500 m inland of the coast. For each date, is the cell **mostly
(more than half) built**: buildings, roads, paved yards, port or industrial structures? Bare ground,
spoil, sand, crops, trees or water are `not_built`. Both dates are side by side: say what each
image shows, not what you expect from the other.

In [ ]:
CELL_ANSWERS = ['built', 'not_built', 'unsure']

def build_cells(cell):
    bbox = square(cell['lon'], cell['lat'], 300)
    ring = cell['outline']['coordinates'][0]
    imgs, answers = [], {}
    for m in REF_DATES:
        img = draw_ring(planet_chip(m, bbox, 'rgb', px=400)[0].copy(), bbox, ring, (0, 255, 255))
        answers[m] = W.RadioButtons(options=CELL_ANSWERS, value=None, description=m)
        imgs.append(W.VBox([W.HTML(f'<b>{m}</b> (image of {PLANET_MONTHS[m]["date"]})'), W.Image(value=png(img), width=400), answers[m]]))

    def collect():
        if any(a.value is None for a in answers.values()):
            raise ValueError('answer both dates')
        return {'cell_id': cell['cell_id'], 'dates': list(REF_DATES), 'months': {m: a.value for m, a in answers.items()},
                'image_dates': {m: PLANET_MONTHS[m]['date'] for m in REF_DATES}}
    return [W.HBox(imgs)], collect

need_planet()
run_form(subset(cells_), out_file('cells'), lambda r: r['cell_id'], build_cells, '(c) cells')

## Task (d) — Month of change at the 30 sites
Every Planet month of the site, in order, with the Phase 3 yellow box. Is there a **lasting** change
inside the box during these months: a land-cover change that is still there in the last image? Floods,
crop cycles, a cloud or a darker image are not lasting changes.
If yes, give the window: the **last month that looks like before** and the **first month that looks
like after**. Months in between that are cloudy just widen the window (this is how CCDC dates breaks,
D14). Most sites changed between 2018 and 2023, so `no` will be a common answer.

In [ ]:
def build_change_month(site):
    ring = site['chip']['coordinates'][0]
    bbox = [min(p[0] for p in ring), min(p[1] for p in ring), max(p[0] for p in ring), max(p[1] for p in ring)]
    outline = site['outline']['coordinates'][0]
    tiles = []
    for m in MONTHS:
        img = draw_ring(planet_chip(m, bbox, 'rgb', px=256)[0].copy(), bbox, outline)
        tiles.append(W.VBox([W.HTML(f'<b>{m}</b> <small>{PLANET_MONTHS[m]["date"]}, clear {PLANET_MONTHS[m]["clear"]:.0%}</small>'),
                             W.Image(value=png(img), width=220)]))
    grid = W.GridBox(tiles, layout=W.Layout(grid_template_columns='repeat(4, 230px)'))
    lasting = W.RadioButtons(options=['yes', 'no', 'unsure'], value=None, description='Lasting change?')
    last_before = W.Dropdown(options=[None] + MONTHS[:-1], description='last before')
    first_after = W.Dropdown(options=[None] + MONTHS[1:], description='first after')

    def collect():
        if lasting.value is None:
            raise ValueError('answer "lasting change?"')
        if lasting.value != 'yes':
            return {'site_id': site['site_id'], 'lasting_change': lasting.value, 'change_window': None, 'months_shown': MONTHS}
        if last_before.value is None or first_after.value is None or last_before.value >= first_after.value:
            raise ValueError('give a window: last month before < first month after')
        return {'site_id': site['site_id'], 'lasting_change': 'yes', 'change_window': [last_before.value, first_after.value],
                'months_shown': MONTHS}
    return [grid, lasting, W.HBox([last_before, first_after])], collect

need_planet()
run_form(subset(sites), out_file('change_month'), lambda r: r['site_id'], build_change_month, '(d) change month')

## Step 4 — Check and bring the labels into the repo
Counts what's done. Then download the `labels14_*.jsonl` files from `MyDrive/evichange/labeling/` into
`data/quangninh/labels/phase14/` and run `pytest tests/test_labels14.py`: `agents/labels14.load_labels()`
checks every record. In the commit that adds them, set `landcover.CLASSES_UNLOCKED_AT_LABELLED_SITES` and
`shoreline.REFERENCE_TRANSECTS_UNLOCKED` to True (after the second labeller has finished too).
**Never edit a label after claims have been produced for its item**; corrections go in a `labels14_v2*` file
with the reason in `notes`.

In [ ]:
expected = {'classes': len(subset(sites)), 'shoreline': len(shore_items), 'cells': len(subset(cells_)), 'change_month': len(subset(sites))}
for task, n in expected.items():
    path = out_file(task)
    rows = [json.loads(l) for l in open(path)] if os.path.exists(path) else []
    print(f'{task:13s} {len(rows):3d} / {n} labelled  ->  {os.path.basename(path)}')